## 15_b_estimation
  Collects the gravity destination exponent b saved by 05/05b and compares it with the scaling
  exponents. No model is refitted. Output: data/processed/15_b_estimation/*.csv.

In [ ]:
# Config block: all names, paths and levels live here
import sys
import os
from pathlib import Path
import pandas as pd

sys.path.append(str(Path.cwd().parent))  # notebook is in notebooks/
from src import config

# Country code -> checkpoint folder of the notebook that saved the fit
PARAM_DIRS = {
    "CZ": "05_gravity_cz",
    "PL": "05b_gravity_pl",
}
PARAMS_FILE = "gravity_params.csv"
REQUIRED_COLS = ["model", "deterrence_func_type", "gamma",
                 "origin_exp", "destination_exp", "gravity_type"]
COL_B = "destination_exp"   # the exponent called b in this notebook

frames = []
for country, folder in PARAM_DIRS.items():
    path = config.PROCESSED_DIR / folder / PARAMS_FILE
    # Fail early with a relative path only (no absolute path in the output)
    if not path.exists():
        raise FileNotFoundError(f"{country}: missing {os.path.relpath(path)} - run the source notebook first")
    df = pd.read_csv(path)
    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    assert not missing, f"{country}: missing columns {missing}"
    df.insert(0, "country", country)
    frames.append(df)
    print(country, "->", os.path.relpath(path), f"({len(df)} rows)")

params = pd.concat(frames, ignore_index=True)
params[["country", "model", "deterrence_func_type", "gamma", "origin_exp", COL_B, "gravity_type"]]

In [ ]:
# Config block: all names, paths and levels live here
BETA_CSV = Path.cwd().parent / "results" / "scaling_by_country" / "beta_by_country_2023.csv"
OUT_DIR = config.PROCESSED_DIR / "15_b_estimation"
B_OUT_CSV = OUT_DIR / "b_values.csv"
BETA_OUT_CSV = OUT_DIR / "beta_values.csv"
SAMPLE = "full"                              # sample of notebook 14 used here (with capitals)
COUNTRIES = sorted(params["country"].unique())   # derived from the loaded b table
BETA_COLS = ["group", "sample", "y_variable", "n_variable", "beta",
             "beta_se_hc3", "beta_ci_low_hc3", "beta_ci_high_hc3", "n_regions"]
B_COLS = ["country", "model", "deterrence_func_type", "gamma", COL_B]

OUT_DIR.mkdir(parents=True, exist_ok=True)
if not BETA_CSV.exists():
    raise FileNotFoundError(f"missing {os.path.relpath(BETA_CSV)} - run notebook 14 first")

beta_all = pd.read_csv(BETA_CSV)
print("samples in file:", sorted(beta_all["sample"].unique()))
print("y variables in file:", sorted(beta_all["y_variable"].unique()))

beta_tbl = beta_all[beta_all["group"].isin(COUNTRIES) & (beta_all["sample"] == SAMPLE)][BETA_COLS]
b_tbl = params[B_COLS]

beta_tbl.to_csv(BETA_OUT_CSV, index=False)
b_tbl.to_csv(B_OUT_CSV, index=False)
print("saved:", os.path.relpath(BETA_OUT_CSV), "|", os.path.relpath(B_OUT_CSV))

display(b_tbl)
display(beta_tbl)

In [ ]:
# Config block: all names, paths and levels live here
COMPARE_VARS = sorted(beta_tbl["y_variable"].unique())   # all variables found in the beta table
COMPARE_CSV = OUT_DIR / "b_vs_beta.csv"

# Keep only the compared variables from the beta table
beta_cmp = beta_tbl[beta_tbl["y_variable"].isin(COMPARE_VARS)][["group", "y_variable", "beta"]]
beta_cmp = beta_cmp.rename(columns={"group": "country", "beta": "beta_scaling"})

# One row per country x deterrence type x variable
cmp_tbl = b_tbl.rename(columns={COL_B: "b_gravity"}).merge(beta_cmp, on="country", how="inner")
cmp_tbl["diff_b_minus_beta"] = cmp_tbl["b_gravity"] - cmp_tbl["beta_scaling"]

cmp_tbl = cmp_tbl[["country", "deterrence_func_type", "y_variable",
                   "b_gravity", "beta_scaling", "diff_b_minus_beta"]]
cmp_tbl = cmp_tbl.sort_values(["country", "y_variable", "deterrence_func_type"]).reset_index(drop=True)

cmp_tbl.to_csv(COMPARE_CSV, index=False)
print("saved:", os.path.relpath(COMPARE_CSV), f"({len(cmp_tbl)} rows)")
print(cmp_tbl.round(3).to_string(index=False))